In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev_ua5816bd", "Catalog Name")
dbutils.widgets.text("bronze_schema", "oles0305_bronze", "Bronze Schema")
dbutils.widgets.text("silver_schema", "oles0305_silver", "Silver Schema")

CATALOG_NAME = dbutils.widgets.get("catalog_name")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE = f"{CATALOG_NAME}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG_NAME}.{SILVER_SCHEMA}"

In [0]:
window_spec = Window.partitionBy(F.col("driverId")).orderBy(F.col("ingestion_timestamp").desc())

df_bronze = ( 
    spark.read.table(f"{BRONZE}.drivers")
        .withColumn("rnk", F.row_number().over(window_spec))
        .filter(F.col("rnk") == 1)
        .drop("rnk", "source_filename", "load_date")
        .withColumnRenamed("ingestion_timestamp", "bronze_ingested_at")
        .withColumn("silver_processed_at", F.current_timestamp())
)

In [0]:
silver_table = DeltaTable.forName(spark, f"{SILVER}.drivers")

df_silver_active = (
    spark.read.table(f"{SILVER}.drivers")
        .filter(F.col("is_active") == True)
)

In [0]:
df_updates_to_existing = (
    df_bronze.alias("new")
        .join(
            df_silver_active.alias("cur"), 
            "driverId"
        )
        .filter(F.col("new.number") != F.col("cur.number"))
        .selectExpr("new.*")
        .withColumn("mergeKey", F.lit(None))
)

df_staged_updates = (
    df_bronze
        .withColumn("mergeKey", F.col("driverId"))
        .unionByName(df_updates_to_existing)
)

In [0]:
(   # SCD Type 2
    silver_table.alias("target")
        .merge(
            df_staged_updates.alias("source"), 
            "target.driverId = source.mergeKey"
        )
        .whenMatchedUpdate(
            condition="target.is_active = true AND target.number != source.number",
            set={
                "is_active": F.lit(False), 
                "valid_to": F.current_timestamp()
            }
        )
        .whenNotMatchedInsert(
            values={
                "driverId": "source.driverId", 
                "driverRef": "source.driverRef", 
                "number": "source.number",
                "code": "source.code", 
                "forename": "source.forename", 
                "surname": "source.surname",
                "dob": "source.dob", 
                "nationality": "source.nationality", 
                "url": "source.url",
                "bronze_ingested_at": "source.bronze_ingested_at",
                "silver_processed_at": "source.silver_processed_at",
                "valid_from": F.current_timestamp(), 
                "valid_to": F.lit(None), 
                "is_active": F.lit(True)
            }
        )
        .execute()
)